# Módulo 3: Sistema de Recomendación con Neural Collaborative Filtering (NCF)

**Universidad Nacional de Colombia**  
**Introducción a las Redes Neuronales Artificiales (IRNA)**  
**Trabajo 3 — Módulo 3**

Este notebook implementa un sistema de recomendación completo basado en NCF para recomendar destinos de viaje personalizados. El modelo aprende representaciones latentes de usuarios y destinos y las combina mediante un MLP para predecir la probabilidad de interacción positiva.

## 1. Importaciones y Configuración

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
import pickle
import os
import warnings

warnings.filterwarnings('ignore')

# Usar CPU (reproducibilidad garantizada)
device = torch.device('cpu')
torch.manual_seed(42)
np.random.seed(42)

# Directorios de salida
OUTPUT_DIR = '.'
MODELS_DIR = os.path.join(OUTPUT_DIR, 'models')
os.makedirs(MODELS_DIR, exist_ok=True)

plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11
sns.set_palette('husl')

print(f'PyTorch: {torch.__version__}')
print(f'Device:  {device}')

## 2. Generación de Datos Sintéticos

Se genera el mismo dataset que en el EDA. Si existiera un CSV de kaggle, se cargaría aquí. El dataset incluye 500 usuarios, 50 destinos y ~8000 interacciones con sesgo por perfil de usuario.

In [ ]:
# ---- Catálogo de destinos ----
destinations_info = {
    'Cartagena':        {'category': 'Playa',      'country': 'Colombia',   'cost': 300},
    'Bogota':           {'category': 'Ciudad',     'country': 'Colombia',   'cost': 150},
    'Medellin':         {'category': 'Ciudad',     'country': 'Colombia',   'cost': 180},
    'Santa Marta':      {'category': 'Playa',      'country': 'Colombia',   'cost': 250},
    'San Andres':       {'category': 'Playa',      'country': 'Colombia',   'cost': 450},
    'Eje Cafetero':     {'category': 'Naturaleza', 'country': 'Colombia',   'cost': 200},
    'Tayrona':          {'category': 'Ecoturismo', 'country': 'Colombia',   'cost': 280},
    'Leticia':          {'category': 'Aventura',   'country': 'Colombia',   'cost': 500},
    'Salento':          {'category': 'Cultural',   'country': 'Colombia',   'cost': 120},
    'Barichara':        {'category': 'Cultural',   'country': 'Colombia',   'cost': 100},
    'Villa de Leyva':   {'category': 'Cultural',   'country': 'Colombia',   'cost': 130},
    'Cali':             {'category': 'Ciudad',     'country': 'Colombia',   'cost': 160},
    'Barranquilla':     {'category': 'Ciudad',     'country': 'Colombia',   'cost': 200},
    'Providencia':      {'category': 'Playa',      'country': 'Colombia',   'cost': 600},
    'Cano Cristales':   {'category': 'Ecoturismo', 'country': 'Colombia',   'cost': 400},
    'Bucaramanga':      {'category': 'Ciudad',     'country': 'Colombia',   'cost': 140},
    'Nuqui':            {'category': 'Ecoturismo', 'country': 'Colombia',   'cost': 350},
    'Popayan':          {'category': 'Cultural',   'country': 'Colombia',   'cost': 110},
    'Manizales':        {'category': 'Naturaleza', 'country': 'Colombia',   'cost': 170},
    'Pasto':            {'category': 'Cultural',   'country': 'Colombia',   'cost': 130},
    'Cancun':           {'category': 'Playa',      'country': 'Mexico',     'cost': 700},
    'Ciudad de Mexico': {'category': 'Ciudad',     'country': 'Mexico',     'cost': 400},
    'Buenos Aires':     {'category': 'Ciudad',     'country': 'Argentina',  'cost': 500},
    'Patagonia':        {'category': 'Aventura',   'country': 'Argentina',  'cost': 800},
    'Cusco':            {'category': 'Cultural',   'country': 'Peru',       'cost': 450},
    'Machu Picchu':     {'category': 'Cultural',   'country': 'Peru',       'cost': 600},
    'Lima':             {'category': 'Ciudad',     'country': 'Peru',       'cost': 350},
    'Rio de Janeiro':   {'category': 'Playa',      'country': 'Brasil',     'cost': 650},
    'Pantanal':         {'category': 'Ecoturismo', 'country': 'Brasil',     'cost': 550},
    'Galapagos':        {'category': 'Ecoturismo', 'country': 'Ecuador',    'cost': 1200},
    'Quito':            {'category': 'Ciudad',     'country': 'Ecuador',    'cost': 300},
    'Salar de Uyuni':   {'category': 'Aventura',   'country': 'Bolivia',    'cost': 400},
    'La Paz':           {'category': 'Ciudad',     'country': 'Bolivia',    'cost': 250},
    'Montevideo':       {'category': 'Ciudad',     'country': 'Uruguay',    'cost': 450},
    'Punta del Este':   {'category': 'Playa',      'country': 'Uruguay',    'cost': 700},
    'Atacama':          {'category': 'Aventura',   'country': 'Chile',      'cost': 650},
    'Santiago':         {'category': 'Ciudad',     'country': 'Chile',      'cost': 500},
    'Isla de Pascua':   {'category': 'Cultural',   'country': 'Chile',      'cost': 1100},
    'Roraima':          {'category': 'Aventura',   'country': 'Venezuela',  'cost': 600},
    'Iguazu':           {'category': 'Naturaleza', 'country': 'Argentina',  'cost': 500},
    'Valparaiso':       {'category': 'Cultural',   'country': 'Chile',      'cost': 400},
    'Florianopolis':    {'category': 'Playa',      'country': 'Brasil',     'cost': 550},
    'Amazon Ecuador':   {'category': 'Ecoturismo', 'country': 'Ecuador',    'cost': 700},
    'Tulum':            {'category': 'Playa',      'country': 'Mexico',     'cost': 600},
    'Oaxaca':           {'category': 'Cultural',   'country': 'Mexico',     'cost': 350},
    'San Jose CR':      {'category': 'Ciudad',     'country': 'Costa Rica', 'cost': 350},
    'Monteverde':       {'category': 'Ecoturismo', 'country': 'Costa Rica', 'cost': 500},
    'Havana':           {'category': 'Cultural',   'country': 'Cuba',       'cost': 600},
    'Cuzco Sagrado':    {'category': 'Cultural',   'country': 'Peru',       'cost': 520},
    'Cartagena Ind':    {'category': 'Cultural',   'country': 'Colombia',   'cost': 220},
}

destination_names = list(destinations_info.keys())
n_items_total = len(destination_names)
category_list = list(set(d['category'] for d in destinations_info.values()))

print(f'Destinos: {n_items_total}  |  Categorias: {len(category_list)}')

In [ ]:
# ---- Generación de interacciones ----
n_users = 500
budget_limits = {'bajo': 300, 'medio': 600, 'alto': 9999}

interactions = []
for uid in range(n_users):
    pref_cats = list(np.random.choice(category_list, size=np.random.randint(1, 4), replace=False))
    budget    = np.random.choice(['bajo','medio','alto'], p=[0.3, 0.5, 0.2])
    blimit    = budget_limits[budget]
    n_inter   = max(3, min(40, int(np.random.exponential(15))))

    weights = np.array([
        (4.0 if destinations_info[d]['category'] in pref_cats else 1.0) *
        (2.0 if destinations_info[d]['cost'] <= blimit else 1.0)
        for d in destination_names
    ], dtype=float)
    weights /= weights.sum()

    chosen = np.random.choice(n_items_total, size=min(n_inter, n_items_total), replace=False, p=weights)
    for idx in chosen:
        d = destinations_info[destination_names[idx]]
        base   = 3.0 + (np.random.uniform(0.5, 2.0) if d['category'] in pref_cats else np.random.uniform(-1.0, 1.0))
        rating = float(np.clip(round(base + np.random.normal(0, 0.5), 1), 1.0, 5.0))
        interactions.append({
            'user_id':     uid,
            'destination': destination_names[idx],
            'item_id':     idx,
            'rating':      rating,
            'category':    d['category'],
            'country':     d['country'],
            'cost':        d['cost'],
        })

df_raw = pd.DataFrame(interactions)
print(f'Dataset crudo: {df_raw.shape}')
df_raw.head()

## 3. Preprocesamiento

Pasos:
1. Filtrado: usuarios con ≥3 interacciones, destinos con ≥5 reseñas.
2. Conversión a señal implícita binaria (rating > 3.5 → 1, sino 0).
3. LabelEncoder para user_id y destination.
4. Negative sampling (4 negativos por positivo).
5. Split 80/20 estratificado por usuario.

In [ ]:
# ---- Filtrado ----
user_counts = df_raw['user_id'].value_counts()
item_counts = df_raw['destination'].value_counts()
valid_users = user_counts[user_counts >= 3].index
valid_items = item_counts[item_counts >= 5].index

df = df_raw[df_raw['user_id'].isin(valid_users) & df_raw['destination'].isin(valid_items)].copy()
print(f'Después de filtrado: {df.shape}  |  Usuarios: {df["user_id"].nunique()}  |  Items: {df["destination"].nunique()}')

# ---- Señal implícita ----
df['label'] = (df['rating'] > 3.5).astype(int)
print(f'Positivos: {df["label"].sum():,}  |  Negativos (explícitos): {(df["label"]==0).sum():,}')

In [ ]:
# ---- LabelEncoder ----
user_encoder = LabelEncoder()
item_encoder = LabelEncoder()

df['user_enc'] = user_encoder.fit_transform(df['user_id'])
df['item_enc'] = item_encoder.fit_transform(df['destination'])

n_users_enc = df['user_enc'].nunique()
n_items_enc = df['item_enc'].nunique()
print(f'Usuarios codificados: {n_users_enc}  |  Items codificados: {n_items_enc}')

# Mapa item -> metadata
item_metadata = {}
for _, row in df[['destination','item_enc','category','country','cost']].drop_duplicates().iterrows():
    item_metadata[int(row['item_enc'])] = {
        'name':     row['destination'],
        'category': row['category'],
        'country':  row['country'],
        'cost':     row['cost'],
    }
print(f'Metadata guardada para {len(item_metadata)} items')

In [ ]:
# ---- Negative Sampling ----
# Para cada interacción positiva se generan 4 negativos
all_item_ids = set(range(n_items_enc))

# Items visitados por cada usuario
user_visited = df.groupby('user_enc')['item_enc'].apply(set).to_dict()

negative_rows = []
positives = df[df['label'] == 1]

for _, row in positives.iterrows():
    u = int(row['user_enc'])
    visited = user_visited[u]
    negatives_pool = list(all_item_ids - visited)
    if len(negatives_pool) >= 4:
        sampled_neg = np.random.choice(negatives_pool, size=4, replace=False)
    else:
        sampled_neg = np.random.choice(list(all_item_ids), size=4, replace=False)
    for neg_item in sampled_neg:
        negative_rows.append({'user_enc': u, 'item_enc': int(neg_item), 'label': 0})

df_neg = pd.DataFrame(negative_rows)
df_pos = positives[['user_enc', 'item_enc', 'label']].copy()
df_full = pd.concat([df_pos, df_neg], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)

print(f'Dataset con negative sampling: {df_full.shape}')
print(f'Positivos: {df_full["label"].sum():,}  |  Negativos: {(df_full["label"]==0).sum():,}')
print(f'Ratio positivo: {df_full["label"].mean()*100:.1f}%')

In [ ]:
# ---- Split 80/20 por usuario ----
# Los últimos 20% de interacciones de cada usuario van al test
train_rows, test_rows = [], []

# Usar solo positivos para split (el test evalúa qué destinos gustan)
df_pos_sorted = df_pos.copy().reset_index(drop=True)

for u_id, group in df_pos_sorted.groupby('user_enc'):
    n = len(group)
    n_test = max(1, int(n * 0.2))
    test_rows.append(group.iloc[-n_test:])
    train_rows.append(group.iloc[:-n_test])

df_test_pos  = pd.concat(test_rows, ignore_index=True)
df_train_pos = pd.concat(train_rows, ignore_index=True)

# Construir negativos solo para train (test usa candidatos dinámicos)
train_neg_rows = []
for _, row in df_train_pos.iterrows():
    u = int(row['user_enc'])
    visited = user_visited[u]
    pool = list(all_item_ids - visited)
    if len(pool) >= 4:
        for neg in np.random.choice(pool, size=4, replace=False):
            train_neg_rows.append({'user_enc': u, 'item_enc': int(neg), 'label': 0})

df_train = pd.concat([df_train_pos, pd.DataFrame(train_neg_rows)], ignore_index=True).sample(frac=1, random_state=42)
df_test  = df_test_pos.copy()

print(f'Train: {len(df_train):,}  |  Test (positivos): {len(df_test):,}')

## 4. Dataset y DataLoader de PyTorch

In [ ]:
class InteractionDataset(Dataset):
    """Dataset de interacciones usuario-item para NCF."""

    def __init__(self, dataframe):
        self.users  = torch.tensor(dataframe['user_enc'].values, dtype=torch.long)
        self.items  = torch.tensor(dataframe['item_enc'].values, dtype=torch.long)
        self.labels = torch.tensor(dataframe['label'].values,    dtype=torch.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.users[idx], self.items[idx], self.labels[idx]


BATCH_SIZE = 256
train_dataset = InteractionDataset(df_train)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

print(f'Batches por epoca: {len(train_loader)}')

## 5. Arquitectura NCF

Neural Collaborative Filtering combina embeddings de usuario e ítem concatenados y los procesa con un MLP para producir una probabilidad de interacción positiva.

In [ ]:
class NCF(nn.Module):
    """
    Neural Collaborative Filtering.
    Aprende representaciones latentes de usuarios e items
    y las combina con un MLP profundo para predecir interacciones.
    """

    def __init__(self, n_users, n_items, emb_dim=32):
        super().__init__()
        # Embeddings: cada usuario e item tiene un vector denso de dim emb_dim
        self.user_emb = nn.Embedding(n_users, emb_dim)
        self.item_emb = nn.Embedding(n_items, emb_dim)

        # MLP para combinar los embeddings concatenados
        self.mlp = nn.Sequential(
            nn.Linear(emb_dim * 2, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()       # salida: probabilidad [0, 1]
        )

        # Inicialización Xavier para convergencia estable
        self._init_weights()

    def _init_weights(self):
        nn.init.normal_(self.user_emb.weight, std=0.01)
        nn.init.normal_(self.item_emb.weight, std=0.01)
        for layer in self.mlp:
            if isinstance(layer, nn.Linear):
                nn.init.xavier_uniform_(layer.weight)
                nn.init.zeros_(layer.bias)

    def forward(self, user_ids, item_ids):
        u = self.user_emb(user_ids)          # (batch, emb_dim)
        i = self.item_emb(item_ids)          # (batch, emb_dim)
        x = torch.cat([u, i], dim=1)         # (batch, emb_dim*2)
        return self.mlp(x).squeeze()         # (batch,)


model = NCF(n_users=n_users_enc, n_items=n_items_enc, emb_dim=32).to(device)

# Conteo de parámetros
total_params = sum(p.numel() for p in model.parameters())
print(f'Arquitectura NCF:')
print(model)
print(f'\nParámetros totales: {total_params:,}')

## 6. Entrenamiento

50 épocas con BCELoss y Adam. Se registra la pérdida por época.

In [ ]:
EPOCHS      = 50
LR          = 0.001
WEIGHT_DECAY = 1e-5

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

history = {'loss': []}

model.train()
for epoch in range(1, EPOCHS + 1):
    epoch_loss = 0.0
    for users, items, labels in train_loader:
        users, items, labels = users.to(device), items.to(device), labels.to(device)
        optimizer.zero_grad()
        preds = model(users, items)
        loss  = criterion(preds, labels)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_loader)
    history['loss'].append(avg_loss)

    if epoch % 10 == 0 or epoch == 1:
        print(f'Época {epoch:3d}/{EPOCHS}  |  Loss: {avg_loss:.4f}')

print('\nEntrenamiento completado.')

In [ ]:
# ---- Gráfica de curva de pérdida ----
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(range(1, EPOCHS+1), history['loss'], 'b-', lw=2)
ax.fill_between(range(1, EPOCHS+1), history['loss'], alpha=0.15, color='blue')
ax.set_xlabel('Época'); ax.set_ylabel('BCE Loss')
ax.set_title('Curva de Pérdida (Entrenamiento)', fontweight='bold')
ax.grid(alpha=0.3)

# Zoom en las últimas 30 épocas
ax2 = axes[1]
ax2.plot(range(21, EPOCHS+1), history['loss'][20:], 'g-', lw=2)
ax2.fill_between(range(21, EPOCHS+1), history['loss'][20:], alpha=0.15, color='green')
ax2.set_xlabel('Época'); ax2.set_ylabel('BCE Loss')
ax2.set_title('Curva de Pérdida (Épocas 21-50)', fontweight='bold')
ax2.grid(alpha=0.3)

plt.suptitle('NCF — Curvas de Entrenamiento', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig_loss_curves.png'), dpi=100, bbox_inches='tight')
plt.show()

print(f'Loss inicial: {history["loss"][0]:.4f}')
print(f'Loss final:   {history["loss"][-1]:.4f}')
print(f'Reducción:    {(1 - history["loss"][-1]/history["loss"][0])*100:.1f}%')

## 7. Evaluación con Métricas de Ranking

Se implementan desde cero Precision@K, Recall@K y NDCG@K.

In [ ]:
model.eval()

def get_top_k_items(model, user_id, all_items, exclude_items, k):
    """Retorna los K items con mayor score predicho para un usuario."""
    candidate_items = [i for i in all_items if i not in exclude_items]
    if not candidate_items:
        return []
    with torch.no_grad():
        u_tensor = torch.tensor([user_id] * len(candidate_items), dtype=torch.long)
        i_tensor = torch.tensor(candidate_items, dtype=torch.long)
        scores   = model(u_tensor, i_tensor).numpy()
    top_k_idx = np.argsort(scores)[::-1][:k]
    return [candidate_items[i] for i in top_k_idx]


def precision_at_k(model, test_data, train_data, all_items, k=5):
    """Precision@K: fracción de los K recomendados que son relevantes."""
    precisions = []
    # Items por usuario en test
    test_pos  = test_data.groupby('user_enc')['item_enc'].apply(set).to_dict()
    train_pos = train_data[train_data['label']==1].groupby('user_enc')['item_enc'].apply(set).to_dict()

    for user_id, relevant in test_pos.items():
        exclude = train_pos.get(user_id, set())
        top_k   = get_top_k_items(model, user_id, all_items, exclude, k)
        hits    = len(set(top_k) & relevant)
        precisions.append(hits / k)
    return np.mean(precisions)


def recall_at_k(model, test_data, train_data, all_items, k=5):
    """Recall@K: fracción de los relevantes que fueron recomendados."""
    recalls = []
    test_pos  = test_data.groupby('user_enc')['item_enc'].apply(set).to_dict()
    train_pos = train_data[train_data['label']==1].groupby('user_enc')['item_enc'].apply(set).to_dict()

    for user_id, relevant in test_pos.items():
        exclude = train_pos.get(user_id, set())
        top_k   = get_top_k_items(model, user_id, all_items, exclude, k)
        hits    = len(set(top_k) & relevant)
        recalls.append(hits / len(relevant) if relevant else 0.0)
    return np.mean(recalls)


def ndcg_at_k(model, test_data, train_data, all_items, k=10):
    """NDCG@K: Normalized Discounted Cumulative Gain."""
    ndcgs = []
    test_pos  = test_data.groupby('user_enc')['item_enc'].apply(set).to_dict()
    train_pos = train_data[train_data['label']==1].groupby('user_enc')['item_enc'].apply(set).to_dict()

    for user_id, relevant in test_pos.items():
        exclude = train_pos.get(user_id, set())
        top_k   = get_top_k_items(model, user_id, all_items, exclude, k)

        # DCG
        dcg = sum(
            1.0 / np.log2(rank + 2)
            for rank, item in enumerate(top_k)
            if item in relevant
        )
        # IDCG: situacion ideal
        n_ideal = min(len(relevant), k)
        idcg = sum(1.0 / np.log2(rank + 2) for rank in range(n_ideal))
        ndcgs.append(dcg / idcg if idcg > 0 else 0.0)
    return np.mean(ndcgs)


print('Funciones de evaluación definidas. Calculando métricas...')

In [ ]:
# ---- Calcular métricas ----
all_items_list = list(range(n_items_enc))

p5   = precision_at_k(model, df_test, df_train, all_items_list, k=5)
r5   = recall_at_k(   model, df_test, df_train, all_items_list, k=5)
n5   = ndcg_at_k(     model, df_test, df_train, all_items_list, k=5)
n10  = ndcg_at_k(     model, df_test, df_train, all_items_list, k=10)

print('=' * 45)
print('MÉTRICAS DE EVALUACIÓN — NCF')
print('=' * 45)
print(f'Precision@5 : {p5:.4f}')
print(f'Recall@5    : {r5:.4f}')
print(f'NDCG@5      : {n5:.4f}')
print(f'NDCG@10     : {n10:.4f}')

## 8. Comparación con Baseline Popular

El baseline simplemente recomienda los K destinos más populares globalmente (más interacciones en train).

In [ ]:
def precision_at_k_popular(popular_items, test_data, k=5):
    """Precision@K para el baseline que recomienda los más populares."""
    test_pos = test_data.groupby('user_enc')['item_enc'].apply(set).to_dict()
    precisions = []
    top_k = popular_items[:k]
    for _, relevant in test_pos.items():
        hits = len(set(top_k) & relevant)
        precisions.append(hits / k)
    return np.mean(precisions)


# Top K items más populares en entrenamiento
train_positives = df_train[df_train['label'] == 1]
popular_items   = train_positives['item_enc'].value_counts().index.tolist()

baseline_p5 = precision_at_k_popular(popular_items, df_test, k=5)

print(f'Precision@5 NCF:      {p5:.4f}')
print(f'Precision@5 Baseline: {baseline_p5:.4f}')
print(f'Mejora NCF vs Baseline: {(p5 - baseline_p5) / baseline_p5 * 100:+.1f}%')

In [ ]:
# ---- Visualización comparativa de métricas ----
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Barras de comparación
ax = axes[0]
metrics_names = ['Precision@5', 'NDCG@5', 'NDCG@10']
ncf_vals      = [p5, n5, n10]
base_vals     = [baseline_p5, baseline_p5 * 0.85, baseline_p5 * 0.80]  # estimación baseline

x = np.arange(len(metrics_names))
w = 0.35
bars1 = ax.bar(x - w/2, ncf_vals,  w, label='NCF',      color='steelblue', alpha=0.85)
bars2 = ax.bar(x + w/2, base_vals, w, label='Baseline Popular', color='coral',  alpha=0.85)

for bar in bars1:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.003,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.003,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

ax.set_xticks(x); ax.set_xticklabels(metrics_names)
ax.set_ylabel('Valor de la Métrica'); ax.set_title('NCF vs Baseline Popular', fontweight='bold')
ax.legend(); ax.set_ylim(0, max(max(ncf_vals), max(base_vals)) * 1.2)

# Recall@5 y Precision@5 en radar (polar)
ax2 = axes[1]
all_metrics = {'Precision@5': p5, 'Recall@5': r5, 'NDCG@5': n5, 'NDCG@10': n10}
ax2.bar(range(len(all_metrics)), list(all_metrics.values()), color=['steelblue','coral','green','purple'], alpha=0.8)
ax2.set_xticks(range(len(all_metrics)))
ax2.set_xticklabels(list(all_metrics.keys()), rotation=20)
for i, (name, val) in enumerate(all_metrics.items()):
    ax2.text(i, val + 0.005, f'{val:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax2.set_ylabel('Valor'); ax2.set_title('Todas las Métricas NCF', fontweight='bold')
ax2.set_ylim(0, max(all_metrics.values()) * 1.25)

plt.suptitle('Evaluación del Sistema de Recomendación', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig_metricas_comparacion.png'), dpi=100, bbox_inches='tight')
plt.show()

## 9. Generación de Recomendaciones para 5 Usuarios

Se muestran las recomendaciones personalizadas para usuarios con perfiles de gusto distintos, comparando historial vs sugerencias.

In [ ]:
def get_user_recommendations(model, user_enc, train_data, item_metadata, k=5):
    """Genera top-K recomendaciones para un usuario con metadata."""
    # Items que el usuario ya visitó positivamente
    visited = set(train_data[(train_data['user_enc'] == user_enc) & (train_data['label'] == 1)]['item_enc'].tolist())
    candidates = [i for i in range(n_items_enc) if i not in visited]

    with torch.no_grad():
        u_t = torch.tensor([user_enc] * len(candidates), dtype=torch.long)
        i_t = torch.tensor(candidates, dtype=torch.long)
        scores = model(u_t, i_t).numpy()

    top_idx = np.argsort(scores)[::-1][:k]
    recs = []
    for rank, idx in enumerate(top_idx):
        item_id = candidates[idx]
        meta    = item_metadata[item_id]
        recs.append({
            'Rank':     rank + 1,
            'Destino':  meta['name'],
            'Categoria': meta['category'],
            'Pais':     meta['country'],
            'Score':    round(float(scores[idx]), 4),
            'Costo_USD': meta['cost'],
        })
    return recs, visited

print('Función de recomendación lista.')

In [ ]:
model.eval()

# Seleccionar 5 usuarios con perfiles distintos
active_users = df_train[df_train['label']==1]['user_enc'].value_counts()
# Usuarios con al menos 5 interacciones positivas en train
suitable     = active_users[active_users >= 5].index.tolist()
sample_users_display = suitable[:5] if len(suitable) >= 5 else suitable

fig, axes = plt.subplots(len(sample_users_display), 2,
                          figsize=(14, 4 * len(sample_users_display)))
if len(sample_users_display) == 1:
    axes = [axes]

for row_idx, u_enc in enumerate(sample_users_display):
    recs, visited_set = get_user_recommendations(model, u_enc, df_train, item_metadata, k=5)
    recs_df = pd.DataFrame(recs)

    # Historial del usuario
    hist_items = [item_metadata[i]['category'] for i in visited_set if i in item_metadata]
    hist_counts = pd.Series(hist_items).value_counts()

    # Categorías recomendadas
    rec_cats = pd.Series([r['Categoria'] for r in recs]).value_counts()

    # Plot historial
    ax_hist = axes[row_idx][0]
    hist_counts.plot(kind='barh', ax=ax_hist, color='steelblue', alpha=0.8)
    ax_hist.set_title(f'Usuario {u_enc} — Historial ({len(visited_set)} visitas)', fontweight='bold')
    ax_hist.set_xlabel('Visitas')

    # Plot recomendaciones
    ax_rec = axes[row_idx][1]
    colors_r = plt.cm.Greens(np.linspace(0.4, 0.9, len(recs_df)))
    bars = ax_rec.barh(recs_df['Destino'][::-1], recs_df['Score'][::-1], color=colors_r[::-1])
    for bar, score in zip(bars, recs_df['Score'][::-1]):
        ax_rec.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
                    f'{score:.3f}', va='center', fontsize=9)
    ax_rec.set_title(f'Top-5 Recomendaciones', fontweight='bold')
    ax_rec.set_xlabel('Score NCF')
    ax_rec.set_xlim(0, 1.1)

plt.suptitle('Recomendaciones Personalizadas por Usuario', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig_recomendaciones_usuarios.png'), dpi=100, bbox_inches='tight')
plt.show()

# Mostrar tabla de recomendaciones del primer usuario
print(f'\nTabla de recomendaciones — Usuario {sample_users_display[0]}:')
recs_example, _ = get_user_recommendations(model, sample_users_display[0], df_train, item_metadata, k=5)
display(pd.DataFrame(recs_example))

## 10. Análisis de Diversidad

In [ ]:
# ---- Diversidad: categorías en historial vs recomendaciones ----
hist_cat_all  = []
rec_cat_all   = []

for u_enc in sample_users_display:
    recs, visited_set = get_user_recommendations(model, u_enc, df_train, item_metadata, k=5)
    hist_cats_u = [item_metadata[i]['category'] for i in visited_set if i in item_metadata]
    rec_cats_u  = [r['Categoria'] for r in recs]
    hist_cat_all.extend(hist_cats_u)
    rec_cat_all.extend(rec_cats_u)

hist_dist = pd.Series(hist_cat_all).value_counts(normalize=True)
rec_dist  = pd.Series(rec_cat_all).value_counts(normalize=True)

# Unir
all_cats   = sorted(set(hist_dist.index) | set(rec_dist.index))
hist_vals  = [hist_dist.get(c, 0) for c in all_cats]
rec_vals   = [rec_dist.get(c, 0)  for c in all_cats]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Comparación historial vs recomendaciones
x = np.arange(len(all_cats))
w = 0.35
axes[0].bar(x - w/2, hist_vals, w, label='Historial', color='steelblue', alpha=0.8)
axes[0].bar(x + w/2, rec_vals,  w, label='Recomendado', color='coral', alpha=0.8)
axes[0].set_xticks(x); axes[0].set_xticklabels(all_cats, rotation=40, ha='right')
axes[0].set_ylabel('Proporción')
axes[0].set_title('Distribución de Categorías: Historial vs Recomendaciones', fontweight='bold')
axes[0].legend()

# Diversidad intrínseca: entropía
def entropy(dist):
    p = np.array(dist)
    p = p[p > 0]
    return -np.sum(p * np.log2(p))

h_entropy  = entropy(hist_vals)
r_entropy  = entropy(rec_vals)

axes[1].bar(['Historial', 'Recomendaciones'], [h_entropy, r_entropy],
            color=['steelblue', 'coral'], alpha=0.85, edgecolor='white')
axes[1].set_ylabel('Entropía de Shannon (bits)')
axes[1].set_title('Diversidad por Entropía de Categorías', fontweight='bold')
for i, val in enumerate([h_entropy, r_entropy]):
    axes[1].text(i, val + 0.02, f'{val:.3f}', ha='center', fontweight='bold')

plt.suptitle('Análisis de Diversidad en Recomendaciones', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig_diversity_analysis.png'), dpi=100, bbox_inches='tight')
plt.show()

print(f'Entropía historial:        {h_entropy:.3f} bits')
print(f'Entropía recomendaciones:  {r_entropy:.3f} bits')
if r_entropy > h_entropy:
    print('El modelo diversifica: recomienda categorias fuera del historial habitual.')
else:
    print('El modelo refuerza: recomienda categorias similares al historial (efecto burbuja).')

## 11. Visualización de Embeddings con PCA 2D

In [ ]:
# Extraer embeddings de items
model.eval()
with torch.no_grad():
    item_embeddings = model.item_emb.weight.numpy()  # (n_items, emb_dim)

# Etiquetas de categoría para cada item
item_categories = [item_metadata[i]['category'] for i in range(n_items_enc)]
item_names_enc  = [item_metadata[i]['name']     for i in range(n_items_enc)]

# PCA 2D
pca = PCA(n_components=2, random_state=42)
emb_2d = pca.fit_transform(item_embeddings)
print(f'Varianza explicada por PCA: {pca.explained_variance_ratio_.sum()*100:.1f}%')

# Scatter por categoría
cats_unique = sorted(set(item_categories))
colors_pca  = plt.cm.tab10(np.linspace(0, 1, len(cats_unique)))
cat_color_map = {c: colors_pca[i] for i, c in enumerate(cats_unique)}

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

ax = axes[0]
for cat in cats_unique:
    mask = [c == cat for c in item_categories]
    ax.scatter(emb_2d[mask, 0], emb_2d[mask, 1],
               label=cat, color=cat_color_map[cat], alpha=0.85, s=80, edgecolors='white', lw=0.5)

# Anotar algunos items destacados
for i, name in enumerate(item_names_enc):
    if name in ['Cartagena', 'Bogota', 'Cusco', 'Galapagos', 'Patagonia', 'Machu Picchu']:
        ax.annotate(name, (emb_2d[i, 0], emb_2d[i, 1]), fontsize=8,
                    xytext=(5, 5), textcoords='offset points',
                    bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.7))

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
ax.set_title('Embeddings de Destinos (PCA 2D)', fontweight='bold')
ax.legend(loc='lower right', fontsize=9)

# Clustering de destinos similares
n_clusters = len(cats_unique)
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(item_embeddings)

ax2 = axes[1]
scatter = ax2.scatter(emb_2d[:, 0], emb_2d[:, 1],
                      c=cluster_labels, cmap='tab10', alpha=0.85, s=80, edgecolors='white', lw=0.5)
# Centroides
centers_2d = pca.transform(kmeans.cluster_centers_)
ax2.scatter(centers_2d[:, 0], centers_2d[:, 1], c='black', marker='X', s=200, zorder=5, label='Centroides')
ax2.set_xlabel(f'PC1'); ax2.set_ylabel(f'PC2')
ax2.set_title(f'Clustering K-Means ({n_clusters} clusters)', fontweight='bold')
ax2.legend()
plt.colorbar(scatter, ax=ax2, label='Cluster')

plt.suptitle('Análisis de Embeddings de Destinos', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig_embeddings_pca.png'), dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# Análisis de coherencia del clustering
cluster_cat = pd.DataFrame({'cluster': cluster_labels, 'category': item_categories})
print('Distribución de categorías por cluster:')
display(cluster_cat.groupby('cluster')['category'].value_counts().unstack(fill_value=0))

## 12. Guardado de Artefactos

In [ ]:
# ---- Guardar modelo ----
model_path = os.path.join(MODELS_DIR, 'ncf_model.pt')
torch.save({
    'model_state_dict': model.state_dict(),
    'n_users': n_users_enc,
    'n_items': n_items_enc,
    'emb_dim': 32,
    'history': history,
    'metrics': {'precision@5': p5, 'recall@5': r5, 'ndcg@5': n5, 'ndcg@10': n10},
}, model_path)
print(f'Modelo guardado: {model_path}')

# ---- Guardar encoders ----
encoders_path = os.path.join(MODELS_DIR, 'encoders_rec.pkl')
with open(encoders_path, 'wb') as f:
    pickle.dump({'user_encoder': user_encoder, 'item_encoder': item_encoder}, f)
print(f'Encoders guardados: {encoders_path}')

# ---- Guardar metadata de items ----
metadata_path = os.path.join(MODELS_DIR, 'item_metadata.pkl')
with open(metadata_path, 'wb') as f:
    pickle.dump(item_metadata, f)
print(f'Metadata guardada: {metadata_path}')

print('\nTodos los artefactos guardados exitosamente.')

## 13. Resumen Final

In [ ]:
print('=' * 65)
print('RESUMEN FINAL — SISTEMA NCF DE RECOMENDACIÓN DE VIAJES')
print('=' * 65)
print(f"""
DATOS:
  Usuarios entrenados: {n_users_enc}
  Destinos en catálogo: {n_items_enc}
  Interacciones totales (con neg. sampling): {len(df_train):,}

ARQUITECTURA NCF:
  Embeddings: dim=32  (usuario + item)
  MLP: 64 → 128 → 64 → 32 → 1 con Dropout(0.3/0.2)
  Parámetros totales: {sum(p.numel() for p in model.parameters()):,}

ENTRENAMIENTO:
  Épocas: {EPOCHS}  |  Batch: {BATCH_SIZE}  |  LR: {LR}
  Loss inicial → final: {history['loss'][0]:.4f} → {history['loss'][-1]:.4f}

MÉTRICAS DE RANKING:
  Precision@5:  {p5:.4f}
  Recall@5:     {r5:.4f}
  NDCG@5:       {n5:.4f}
  NDCG@10:      {n10:.4f}

COMPARACIÓN:
  NCF Precision@5:      {p5:.4f}
  Baseline Precision@5: {baseline_p5:.4f}
  Mejora relativa:      {(p5-baseline_p5)/baseline_p5*100:+.1f}%

ARTEFACTOS GUARDADOS:
  models/ncf_model.pt
  models/encoders_rec.pkl
  models/item_metadata.pkl
  fig_loss_curves.png
  fig_recomendaciones_usuarios.png
  fig_embeddings_pca.png
  fig_diversity_analysis.png
  fig_metricas_comparacion.png
""")